# SnowJev Infrastructure Setup

This notebook creates all the infrastructure needed for the SnowJev (Decider) demo:
1. Database & schema
2. Network rules & external access integration (`SNOWJEV_EXTERNAL_ACCESS`)
3. PyPI access integration (`PYPI_ACCESS_INTEGRATION`)
4. Instructions to connect `snow_jev_decider_demo.ipynb` to a GPU notebook service

Uses the system-provided `SYSTEM_COMPUTE_POOL_GPU` (`GPU_NV_S` / A10G — 28 GiB RAM, 450 GB NVMe, 24 GB VRAM).

Run cells in order. Each step is idempotent (safe to re-run).

## 1. Database & Schema

In [ ]:
%%sql -r infra_db
CREATE DATABASE IF NOT EXISTS SNOWJEV;
CREATE SCHEMA IF NOT EXISTS SNOWJEV.DECIDER;
USE SCHEMA SNOWJEV.DECIDER;

## 2. Network Rules & EAI

Grants egress to HuggingFace (model download) and GitHub.

**Hosts included:**
- `huggingface.co` and CDN subdomains (`*.hf.co`, `*.cdn.hf.co`, `*.aws.cdn.hf.co`)
- `github.com`

In [ ]:
%%sql -r infra_net_rule
-- Network rule: GitHub + HuggingFace (including multi-level CDN subdomains)
CREATE OR REPLACE NETWORK RULE SNOWJEV.DECIDER.EXTERNAL_NETWORK_RULE
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = (
    'github.com',
    'huggingface.co',
    '*.huggingface.co',
    '*.hf.co',
    '*.cdn.hf.co',
    '*.aws.cdn.hf.co'
  );

In [ ]:
%%sql -r infra_eai
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION SNOWJEV_EXTERNAL_ACCESS
  ALLOWED_NETWORK_RULES = (SNOWJEV.DECIDER.EXTERNAL_NETWORK_RULE)
  ENABLED = TRUE;

DESCRIBE EXTERNAL ACCESS INTEGRATION SNOWJEV_EXTERNAL_ACCESS;

## 3. PyPI Access Integration

Uses the Snowflake-managed PyPI network rule for `pip install` inside the notebook service.

In [ ]:
%%sql -r infra_pypi
CREATE EXTERNAL ACCESS INTEGRATION IF NOT EXISTS PYPI_ACCESS_INTEGRATION
  ALLOWED_NETWORK_RULES = (SNOWFLAKE.EXTERNAL_ACCESS.PYPI_RULE)
  ENABLED = TRUE;

DESCRIBE EXTERNAL ACCESS INTEGRATION PYPI_ACCESS_INTEGRATION;

## 4. Connect Demo Notebook to GPU Service

After running the cells above, open `snow_jev_decider_demo.ipynb` and connect it to a **notebook service** on `SYSTEM_COMPUTE_POOL_GPU`:

1. Open `snow_jev_decider_demo.ipynb` in this workspace
2. Click **Notebook actions** (top-right) > **Notebook settings** or **Edit Service**
3. Select **Runtime**: GPU 2.10 (Python 3.12)
4. Select **Compute Pool**: `SYSTEM_COMPUTE_POOL_GPU`
5. Click **Save** / **Create**

EAIs are attached automatically by the first SQL cell in the demo notebook.

The notebook service will provision on the compute pool (~2-4 min). `GPU_NV_S` provides 28 GiB RAM, 450 GB NVMe, and 24 GB GPU VRAM (A10G) — enough for the 8.4 GB model download and inference.

## 5. Verification

Confirm everything was created correctly.

In [ ]:
%%sql -r verify_pool
-- Verify system GPU compute pool exists
DESCRIBE COMPUTE POOL SYSTEM_COMPUTE_POOL_GPU;

In [ ]:
-- Verify both external access integrations
DESCRIBE EXTERNAL ACCESS INTEGRATION SNOWJEV_EXTERNAL_ACCESS;
DESCRIBE EXTERNAL ACCESS INTEGRATION PYPI_ACCESS_INTEGRATION;

In [ ]:
%%sql -r verify_rule
-- Verify network rule
DESCRIBE NETWORK RULE SNOWJEV.DECIDER.EXTERNAL_NETWORK_RULE;

In [ ]:
-- Verify services in SNOWJEV.DECIDER
SHOW SERVICES IN SCHEMA SNOWJEV.DECIDER;

## Cleanup (optional)

Uncomment and run to tear down all infrastructure.

In [ ]:
%%sql -r cleanup
/*
 DROP SERVICE IF EXISTS SNOWJEV.DECIDER.DECIDER_SERVICE;
 DROP EXTERNAL ACCESS INTEGRATION IF EXISTS SNOWJEV_EXTERNAL_ACCESS;
 DROP EXTERNAL ACCESS INTEGRATION IF EXISTS PYPI_ACCESS_INTEGRATION;
 DROP DATABASE IF EXISTS SNOWJEV CASCADE;
 */